# Tutorial: Auditoria de Gradient Masking

**Público:** avaliadores de robustez adversarial.

**Objetivos:** executar checks independentes e interpretar sinais de falsa robustez.

## Roteiro

1. Preparar modelo e dados.
2. Comparar FGSM/PGD.
3. Avaliar curvas de epsilon, steps e restarts.
4. Exercício: introduzir uma defesa não diferenciável.

> Pitfall: ataque fraco não prova defesa forte.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "python"))
import torch
from torch import nn
from ml_security.gradient_masking_audit import GradientMaskingAuditor

model = nn.Linear(2, 2)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[-10., -10.], [10., 10.]]))
    model.bias.copy_(torch.tensor([10., -10.]))
x = torch.tensor([[.20, .20], [.25, .30], [.80, .80], [.70, .75]])
y = torch.tensor([0, 0, 1, 1])

## Auditoria

O relatório não “prova ausência” de masking; agrega inconsistências úteis para investigação e explicita BPDA/EOT.

In [2]:
report = GradientMaskingAuditor().audit(
    model, x, y,
    epsilon=0.3,
    alpha=0.08,
    epsilon_values=(0.0, 0.1, 0.2, 0.3),
    step_values=(1, 5, 10, 20),
    restart_values=(1, 3, 5),
)
report

{'fgsm_robust_accuracy': 0.25,
 'pgd_robust_accuracy': 0.25,
 'transfer_robust_accuracy': None,
 'blackbox_robust_accuracy': None,
 'epsilon_curve': [{'epsilon': 0.0, 'robust_accuracy': 1.0},
  {'epsilon': 0.1, 'robust_accuracy': 1.0},
  {'epsilon': 0.2, 'robust_accuracy': 1.0},
  {'epsilon': 0.3, 'robust_accuracy': 0.25}],
 'steps_curve': [{'steps': 1, 'robust_accuracy': 1.0},
  {'steps': 5, 'robust_accuracy': 0.25},
  {'steps': 10, 'robust_accuracy': 0.25},
  {'steps': 20, 'robust_accuracy': 0.25}],
 'restart_curve': [{'restarts': 1, 'robust_accuracy': 0.25},
  {'restarts': 3, 'robust_accuracy': 0.25},
  {'restarts': 5, 'robust_accuracy': 0.25}],
 'checks_executed': 5,
 'bpda_required': False,
 'eot_required': False,
 'masking_risk_score': 0,
 'risk': 'LOW',
 'flags': []}

## Exercício

Adicione arredondamento da entrada antes do modelo e marque bpda_required. Discuta por que gradiente zero/instável pode enganar a avaliação.

**Extensão:** forneça exemplos por transferência e black-box ao auditor para executar seis checks.

In [3]:
{
    "risk": report["risk"],
    "flags": report["flags"],
    "checks": report["checks_executed"],
    "epsilon_curve": report["epsilon_curve"],
}

{'risk': 'LOW',
 'flags': [],
 'checks': 5,
 'epsilon_curve': [{'epsilon': 0.0, 'robust_accuracy': 1.0},
  {'epsilon': 0.1, 'robust_accuracy': 1.0},
  {'epsilon': 0.2, 'robust_accuracy': 1.0},
  {'epsilon': 0.3, 'robust_accuracy': 0.25}]}